# Phase 3 — KPI Calculations

This section calculates the core business KPIs requested in the project brief, using the cleaned
IndiaKart data.

In [5]:
import pandas as pd
import numpy as np

In [2]:
files = {
    'orders':      'orders DA.csv',
    'order_items': 'order_items DA.csv',
    'customers':   'customers DA.csv',
    'products':    'products DA.csv',
    'payments':    'payments DA.csv',
    'returns':     'returns DA.csv',
    'inventory':   'inventory DA.csv',
    'suppliers':   'suppliers DA.csv',
}

In [3]:
dfs = {name: pd.read_csv(path) for name, path in files.items()}
orders      = dfs['orders']
order_items = dfs['order_items']
customers   = dfs['customers']
products    = dfs['products']
payments    = dfs['payments']
returns     = dfs['returns']
inventory   = dfs['inventory']
suppliers   = dfs['suppliers']

In [4]:
orders['order_date']     = pd.to_datetime(orders['order_date'], format='%d-%m-%Y')
orders['delivered_date'] = pd.to_datetime(orders['delivered_date'], format='%d-%m-%Y', errors='coerce')

### KPI 1: GMV (Gross Merchandise Value)

In [6]:
gmv = orders['final_amount'].sum()
print(f"GMV: Rs.{gmv:,.2f}")

GMV: Rs.3,147,385,544.58


### KPI 2: Net Revenue

In [8]:
orders.dtypes

order_id                    object
customer_id                 object
order_date          datetime64[ns]
order_time                  object
status                      object
city                        object
state                       object
pincode                      int64
total_amount               float64
gst_amount                 float64
shipping_charge              int64
discount_amount            float64
final_amount               float64
payment_method              object
shipping_partner            object
tracking_id                 object
delivered_date      datetime64[ns]
is_cod                       int64
channel                     object
dtype: object

In [12]:
delivered_orders = orders[orders['status'] == 'Delivered']
print(delivered_orders.shape)

(32499, 19)


In [13]:
net_revenue = delivered_orders['final_amount'].sum()
print(f"Net Revenue: Rs.{net_revenue:,.2f}")

Net Revenue: Rs.2,053,858,611.96


In [14]:
pct_of_gmv = net_revenue / gmv * 100
print(f"Net Revenue as % of GMV: {pct_of_gmv:.2f}%")

Net Revenue as % of GMV: 65.26%


### KPI 3: Average Order Value (AOV)

In [15]:
Count_of_Delivered_Orders = len(delivered_orders)

In [17]:
aov = net_revenue/Count_of_Delivered_Orders
print(f"AOV: Rs.{aov:,.2f}")

AOV: Rs.63,197.59


### KPI 4: Cancellation Rate

In [23]:
cancelled_orders = orders[orders['status'] == 'Cancelled']

In [22]:
total_orders = total_orders = len(orders)

In [25]:
cancellation_rate = (len(cancelled_orders)/total_orders) * 100
print(f"Cancellation Rate: {cancellation_rate:.2f}%")

Cancellation Rate: 11.99%


### KPI 5: Return Rate

In [30]:
print("Total rows in returns.csv:", len(returns))
print("Unique order_ids in returns.csv:", returns['order_id'].nunique())

Total rows in returns.csv: 10000
Unique order_ids in returns.csv: 10000


In [31]:
returned_order_ids = returns['order_id']
orders[orders['order_id'].isin(returned_order_ids)]['status'].value_counts()

status
Delivered    4327
Returned     3933
Cancelled    1067
Shipped       673
Name: count, dtype: int64

In [34]:
return_rate = (return_count/Count_of_Delivered_Orders)*100
print(f"Return Rate: {return_rate:.2f}%")

Return Rate: 30.77%


In [35]:
# orders that were ever actually delivered (currently Delivered OR Returned)
ever_delivered_count = Count_of_Delivered_Orders + len(orders[orders['status'] == 'Returned'])

# valid returns = return records where the order was genuinely delivered/returned (not Cancelled/Shipped)
valid_return_count = len(orders[orders['order_id'].isin(returned_order_ids) & orders['status'].isin(['Delivered', 'Returned'])])

return_rate = valid_return_count / ever_delivered_count * 100
print(f"Valid returns: {valid_return_count}")
print(f"Ever-delivered orders: {ever_delivered_count}")
print(f"Return Rate (corrected): {return_rate:.2f}%")

Valid returns: 8260
Ever-delivered orders: 36432
Return Rate (corrected): 22.67%


### KPI 6: Customer Lifetime Value (CLV)

In [39]:
clv_by_segment = customers.groupby('segment')['total_spent'].mean().sort_values(ascending=False)
print(clv_by_segment)

segment
Premium     532734.995816
Regular     361870.720984
New         274866.478952
Budget      186178.807764
Inactive     32249.081354
Name: total_spent, dtype: float64


In [40]:
ratio = clv_by_segment['Premium'] / clv_by_segment['Regular']
print(f"Premium is {ratio:.2f}x Regular")

Premium is 1.47x Regular


### KPI 7: Month-over-Month (MoM) Growth

In [42]:
mom_growth_clean = mom_growth.iloc[2:-1]  # drop first two rows and the last row
print(mom_growth_clean)
print(f"\nAverage MoM growth (excluding partial months): {mom_growth_clean.mean():.2f}%")

order_date
2023-08    -0.619352
2023-09     7.290273
2023-10    -3.631252
2023-11     3.617765
2023-12    -0.247668
2024-01    -3.997401
2024-02    -7.990990
2024-03    10.520565
2024-04    -1.616161
2024-05     3.686903
2024-06    -9.253108
2024-07     2.656586
2024-08     3.853626
2024-09    -1.059846
2024-10     2.293093
2024-11    -5.433110
2024-12     5.405587
2025-01     3.636394
2025-02   -17.264213
2025-03    19.608423
2025-04    -4.154785
2025-05     3.842741
Freq: M, Name: final_amount, dtype: float64

Average MoM growth (excluding partial months): 0.51%


### KPI 8: Top Category Revenue Share

In [43]:
category_revenue = order_items.groupby('category')['total_price'].sum().sort_values(ascending=False)
category_share = category_revenue / category_revenue.sum() * 100
print(category_share)

top_category_share = category_share.iloc[0]
print(f"\nTop category ({category_share.index[0]}) share: {top_category_share:.2f}%")

category
Electronics         57.409452
Sports & Fitness    17.380534
Home & Kitchen       8.539124
Fashion              5.178639
Automotive           4.316313
Office Supplies      3.251760
Toys & Baby          1.628298
Beauty & Health      1.154119
Grocery              0.654295
Books                0.487467
Name: total_price, dtype: float64

Top category (Electronics) share: 57.41%


### KPI 9: Payment Failure Rate

In [44]:
payments.dtypes

payment_id         object
order_id           object
customer_id        object
payment_date       object
payment_time       object
payment_method     object
amount            float64
status             object
transaction_id     object
bank_name          object
gateway            object
refund_amount       int64
refund_date       float64
dtype: object

In [46]:
failed_payments = payments[payments['status'] == "Failed"]

In [47]:
print(len(failed_payments))

1748


In [48]:
payment_failure_pct = (len(failed_payments)/len(payments))*100

In [52]:
print(f"{payment_failure_pct:.2f}%")

3.50%


### KPI 10: Inventory Fill Rate

In [53]:
inventory.dtypes

inventory_id             object
product_id               object
warehouse_location       object
quantity_available        int64
quantity_reserved         int64
reorder_level             int64
reorder_quantity          int64
last_restocked_date      object
unit_cost                 int64
total_inventory_value     int64
status                   object
dtype: object

In [54]:
in_stock_sku = inventory[inventory['status'] == "In Stock"]


In [55]:
len(in_stock_sku)

922

In [56]:
inventory_fill_rate = (len(in_stock_sku))/(len(inventory)) * 100
print(f"{inventory_fill_rate:.2f}%")

92.20%


## Phase 3 — KPI Summary

Calculated all 10 KPIs the brief asked for, using the cleaned data from Phase 1. Short version:
**6 out of 10 KPIs miss their target.** This isn't a couple of isolated issues — cancellations,
returns, payment failures, and category concentration all point toward the same underlying
story: the business is stable in size, but leaking value at several points along the way.

| KPI | Value | Target | Status |
|---|---|---|---|
| GMV | ₹314.7 crore | — | Total business size, all orders |
| Net Revenue | ₹205.4 crore (65.26% of GMV) | — | Actual earned revenue, Delivered only |
| AOV | ₹63,197.59 | — | Average value per delivered order |
| Cancellation Rate | 11.99% | <10% | ❌ Misses |
| Return Rate | 22.67% (corrected) | <8% | ❌ Misses, badly |
| CLV — Premium vs Regular | 1.47x | 3x | ❌ Well short |
| MoM Growth (avg) | 0.51% | — | Essentially flat, no real trend |
| Top Category Share | 57.41% (Electronics) | <60% (risk) | ⚠️ Close to risk line |
| Payment Failure Rate | 3.50% | <2% | ❌ Misses |
| Inventory Fill Rate | 92.20% | >95% | ❌ Misses |

### Notes on a few of these

**Net Revenue vs GMV** — only 65.26% of total order value actually turns into real revenue. That
lines up almost exactly with the fact that only 65% of orders end up Delivered — the two numbers
are really telling the same story from different angles.

**Return Rate** — this one needed real digging. A naive calculation (all 10,000 return records ÷
delivered orders) gives 30.77%, which looked too extreme to trust. Checking further, 1,740 of
those return records belong to orders that are currently Cancelled or Shipped — which shouldn't
be possible, since you can't return something that was never delivered. Excluding those and
fixing the denominator (to include orders currently Returned, since they were delivered before
they were returned) gives a corrected rate of 22.67%. Still nearly 3x over target, but now it's a
number that's actually defensible.

**CLV** — Premium customers are the most valuable segment by a clear margin (₹5,32,735 avg lifetime
spend vs Regular's ₹3,61,871), but at 1.47x Regular, they fall well short of the 3x benchmark the
brief expected. Combined with Premium also being the smallest segment by headcount (from Phase 2),
this points to a real growth opportunity — there's room to both grow the Premium segment and get
more value out of the Premium customers already there.

**MoM Growth** — bounces around a lot month to month (from -17% to +19%), but averages out to
just 0.51%. Basically flat. Matches what showed up earlier in the EDA — no clear growth or decline
trend across the full 24 months.

**Top Category Share** — Electronics sits at 57.41%, just under the 60% concentration-risk line,
but close enough that it's still worth flagging. Nothing else comes close to replacing it if
Electronics demand ever dropped.

### Bottom line

GMV and revenue are stable, but stability isn't the same as health. Six KPIs missing target — most
of them by a meaningful margin — means there's real work to do on order fulfillment (cancellations),
product/logistics quality (returns), payment reliability, and growing the Premium segment. These are
the numbers to lead with in the Phase 4 management report.